[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/E.%20Integrated%2C%20Resilient%2C%20%26%20Modern%20Supply%20Chains%20%28The%20Frontier%29/097.%20The%20Last-Mile%20Delivery%20%26%20Micro-Fulfillment%20Problem/P97-Tier-1_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---



# 97. The Last-Mile Delivery & Micro-Fulfillment Problem
## Tier 1: The Pen & Paper Method (Mathematical Formulation)

### Key assumptions
- Delivery requests have known weights/volumes and time windows
- Vehicles have fixed capacities and operating costs
- Micro-fulfillment centers serve as depots for vehicle routing
- Distance costs are proportional to actual travel distances
- Time window violations incur penalty costs

### Approach (step-by-step)
1. **Problem Formulation**: Model as mixed-integer programming with binary assignment variables
2. **Objective Function**: Minimize total cost including distance, time, and penalties
3. **Constraints**: Ensure assignment, capacity, sourcing, and timing requirements
4. **Solution Method**: Use exact optimization with pulp solver
5. **Analysis**: Sensitivity analysis on key parameters

### What to look for in the results
- Optimal assignment of delivery requests to vehicles and fulfillment centers
- Vehicle capacity utilization rates
- Total cost breakdown by component (distance, time, penalties)
- On-time delivery performance metrics

### Concrete example (from the source)
We'll implement the example with 8 delivery requests, 3 vehicles, and 3 micro-fulfillment centers as described in the problem statement.

In [1]:
from typing import Tuple, List, Dict, Set

# Import required libraries for mathematical optimization
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pulp import *
from dataclasses import dataclass
from typing import List, Tuple, Dict
import warnings
warnings.filterwarnings('ignore')

# Set style for visualizations
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

In [ ]:
@dataclass
class DeliveryRequest:
    """Represents a delivery request with weight, location, and time window"""
    id: int
    weight: float  # kg
    location: Tuple[float, float]  # (x, y) coordinates
    time_window: Tuple[int, int]  # (start, end) in hours

@dataclass
class Vehicle:
    """Represents a delivery vehicle with capacity and operating costs"""
    id: int
    capacity: float  # kg
    hourly_cost: float  # cost per operating hour
    home_depot: int  # micro-fulfillment center ID

@dataclass
class MicroFulfillmentCenter:
    """Represents a micro-fulfillment center location"""
    id: int
    location: Tuple[float, float]  # (x, y) coordinates

def calculate_distance(loc1: Tuple[float, float], loc2: Tuple[float, float]) -> float:
    """Calculate Euclidean distance between two locations"""
    return np.sqrt((loc1[0] - loc2[0])**2 + (loc1[1] - loc2[1])**2)

In [ ]:
# Initialize the concrete example from the problem statement
# 8 delivery requests with specified weights and locations
requests = [
    DeliveryRequest(1, 2.5, (20, 30), (8, 12)),   # Request 1
    DeliveryRequest(2, 1.8, (35, 25), (9, 11)),   # Request 2
    DeliveryRequest(3, 3.2, (15, 40), (10, 14)),  # Request 3
    DeliveryRequest(4, 1.5, (40, 35), (8, 10)),   # Request 4
    DeliveryRequest(5, 2.8, (25, 15), (11, 15)),  # Request 5
    DeliveryRequest(6, 2.1, (30, 45), (12, 16)),   # Request 6
    DeliveryRequest(7, 1.9, (45, 20), (9, 13)),   # Request 7
    DeliveryRequest(8, 3.5, (10, 25), (13, 17))    # Request 8
]

# 3 vehicles with different capacities
vehicles = [
    Vehicle(1, 12.0, 25.0, 1),  # Vehicle 1: 12kg capacity, $25/hour, from MFC1
    Vehicle(2, 10.0, 20.0, 2),  # Vehicle 2: 10kg capacity, $20/hour, from MFC2
    Vehicle(3, 14.0, 30.0, 3)   # Vehicle 3: 14kg capacity, $30/hour, from MFC3
]

# 3 micro-fulfillment centers at strategic locations
fulfillment_centers = [
    MicroFulfillmentCenter(1, (25, 30)),  # MFC1 at center
    MicroFulfillmentCenter(2, (15, 20)),  # MFC2 at southwest
    MicroFulfillmentCenter(3, (40, 40))   # MFC3 at northeast
]

print(f"Initialized {len(requests)} delivery requests, {len(vehicles)} vehicles, and {len(fulfillment_centers)} micro-fulfillment centers")

Initialized 8 delivery requests, 3 vehicles, and 3 micro-fulfillment centers


In [ ]:
# Create the mathematical optimization model
def create_optimization_model(requests: List[DeliveryRequest], 
                             vehicles: List[Vehicle],
                             fulfillment_centers: List[MicroFulfillmentCenter]) -> LpProblem:
    """Create mixed-integer programming model for last-mile delivery"""
    
    # Create model instance
    model = LpProblem("Last_Mile_Delivery_Optimization", LpMinimize)
    
    # Cost coefficients
    alpha = 1.0  # Distance cost coefficient
    beta = 1.0   # Time cost coefficient  
    gamma = 10.0 # Penalty cost coefficient for late deliveries
    
    # Decision variables
    # x[i][v] = 1 if request i assigned to vehicle v
    x = {}
    for i, request in enumerate(requests):
        for v, vehicle in enumerate(vehicles):
            x[(i, v)] = LpVariable(f"x_{i}_{v}", cat='Binary')
    
    # z[i][f] = 1 if request i sourced from fulfillment center f
    z = {}
    for i, request in enumerate(requests):
        for f, center in enumerate(fulfillment_centers):
            z[(i, f)] = LpVariable(f"z_{i}_{f}", cat='Binary')
    
    # u[i][t] = 1 if request i delivered in time period t
    time_periods = 8  # 8 time periods (4-hour periods over 32 hours)
    u = {}
    for i, request in enumerate(requests):
        for t in range(time_periods):
            u[(i, t)] = LpVariable(f"u_{i}_{t}", cat='Binary')
    
    # y[v][t] = 1 if vehicle v operates in time period t
    y = {}
    for v, vehicle in enumerate(vehicles):
        for t in range(time_periods):
            y[(v, t)] = LpVariable(f"y_{v}_{t}", cat='Binary')
    
    # Objective function components
    distance_cost = 0
    time_cost = 0
    penalty_cost = 0
    
    # Calculate distance cost (requests to fulfillment centers)
    for i, request in enumerate(requests):
        for v, vehicle in enumerate(vehicles):
            for f, center in enumerate(fulfillment_centers):
                distance = calculate_distance(request.location, center.location)
                distance_cost += alpha * distance * x[(i, v)] * z[(i, f)]
    
    # Calculate time cost (vehicle operating costs)
    for v, vehicle in enumerate(vehicles):
        for t in range(time_periods):
            time_cost += beta * vehicle.hourly_cost * y[(v, t)]
    
    # Calculate penalty cost for late deliveries
    for i, request in enumerate(requests):
        for t in range(time_periods):
            # Penalty if delivered after the end of time window
            if t * 4 > request.time_window[1]:  # Convert periods to hours
                penalty_cost += gamma * (t * 4 - request.time_window[1]) * u[(i, t)]
    
    # Set objective function
    model += distance_cost + time_cost + penalty_cost, "Total_Cost_Minimization"
    
    # Add constraints
    
    # Constraint 1: Each request must be assigned to exactly one vehicle
    for i, request in enumerate(requests):
        model += lpSum(x[(i, v)] for v in range(len(vehicles))) == 1, f"assign_request_{i}"
    
    # Constraint 2: Vehicle capacity constraints
    for v, vehicle in enumerate(vehicles):
        model += lpSum(requests[i].weight * x[(i, v)] for i in range(len(requests))) <= vehicle.capacity, f"capacity_vehicle_{v}"
    
    # Constraint 3: Each request must be sourced from exactly one fulfillment center
    for i, request in enumerate(requests):
        model += lpSum(z[(i, f)] for f in range(len(fulfillment_centers))) == 1, f"source_request_{i}"
    
    # Constraint 4: Each request must be delivered in exactly one time period
    for i, request in enumerate(requests):
        model += lpSum(u[(i, t)] for t in range(time_periods)) == 1, f"delivery_time_{i}"
    
    # Constraint 5: Time window constraints (no delivery before start time)
    for i, request in enumerate(requests):
        for t in range(time_periods):
            if t * 4 < request.time_window[0]:  # Convert periods to hours
                model += u[(i, t)] == 0, f"time_window_start_{i}_{t}"
    
    # Constraint 6: Vehicle must operate if it has assigned deliveries
    for v, vehicle in enumerate(vehicles):
        for t in range(time_periods):
            model += y[(v, t)] >= lpSum(x[(i, v)] * u[(i, t)] for i in range(len(requests))) / len(requests), f"vehicle_operation_{v}_{t}"
    
    return model

print("Optimization model creation function defined")

Optimization model creation function defined


In [ ]:
try:
    # Solve the optimization model
    model = create_optimization_model(requests, vehicles, fulfillment_centers)
    
    print("Solving optimization model...")
    model.solve(pulp.PULP_CBC_CMD(msg=False))
    
    # Check solution status
    if LpStatus[model.status] == 'Optimal':
        print("✅ Optimal solution found!")
        print(f"Total Cost: ${model.objective.value:.2f}")
    else:
        print(f"❌ Solution status: {LpStatus[model.status]]")
except Exception as e:
    print(f'Error in cell: {e}')

[Error handled: f-string: unmatched ']' (<string>, line 12)...]

In [ ]:
try:
    # Extract and analyze the solution
    def extract_solution(model, requests, vehicles, fulfillment_centers):
        """Extract solution details from the optimization model"""
        
        solution = {
            'assignments': [],
            'vehicle_loads': {v.id: 0 for v in vehicles},
            'center_usage': {f.id: 0 for f in fulfillment_centers},
            'delivery_times': {},
            'cost_breakdown': {'distance': 0, 'time': 0, 'penalty': 0}
        }
        
        # Extract assignments
        for i, request in enumerate(requests):
            for v, vehicle in enumerate(vehicles):
                if 'x' in model.variablesDict():
                    x_var = model.variablesDict()[f'x_{i}_{v}']
                    if x_var.value() == 1:
                        # Find fulfillment center
                        for f, center in enumerate(fulfillment_centers):
                            z_var = model.variablesDict()[f'z_{i}_{f}']
                            if z_var.value() == 1:
                                solution['assignments'].append({
                                    'request_id': request.id,
                                    'vehicle_id': vehicle.id,
                                    'center_id': center.id,
                                    'weight': request.weight,
                                    'distance': calculate_distance(request.location, center.location)
                                })
                                solution['vehicle_loads'][vehicle.id] += request.weight
                                solution['center_usage'][center.id] += 1
                                solution['cost_breakdown']['distance'] += calculate_distance(request.location, center.location)
                                break
                        
                        # Find delivery time
                        for t in range(8):  # 8 time periods
                            u_var = model.variablesDict()[f'u_{i}_{t}']
                            if u_var.value() == 1:
                                solution['delivery_times'][request.id] = t * 4  # Convert to hours
                                # Calculate penalty if late
                                if t * 4 > request.time_window[1]:
                                    solution['cost_breakdown']['penalty'] += 10 * (t * 4 - request.time_window[1])
                                break
        
        # Calculate time costs
        for v, vehicle in enumerate(vehicles):
            for t in range(8):
                if 'y' in model.variablesDict():
                    y_var = model.variablesDict()[f'y_{v}_{t}']
                    if y_var.value() == 1:
                        solution['cost_breakdown']['time'] += vehicle.hourly_cost
        
        return solution
    
    # Extract solution
    solution = extract_solution(model, requests, vehicles, fulfillment_centers)
    
    print("Solution extracted successfully!")
    print(f"Total assignments: {len(solution['assignments'])}")
except Exception as e:
    print(f'Error in cell: {e}')

[Error handled: name 'model' is not defined...]

In [ ]:
try:
    # Display solution summary
    print("=" * 60)
    print("OPTIMAL SOLUTION SUMMARY")
    print("=" * 60)
    
    # Display assignments by vehicle
    for vehicle_id in [1, 2, 3]:
        vehicle_assignments = [a for a in solution['assignments'] if a['vehicle_id'] == vehicle_id]
        if vehicle_assignments:
            total_weight = sum(a['weight'] for a in vehicle_assignments)
            total_distance = sum(a['distance'] for a in vehicle_assignments)
            capacity_utilization = (total_weight / [v.capacity for v in vehicles if v.id == vehicle_id][0]) * 100
            
            print(f"\n🚚 Vehicle {vehicle_id}:")
            print(f"   Assigned requests: {[a['request_id'] for a in vehicle_assignments]}")
            print(f"   Total weight: {total_weight:.1f} kg")
            print(f"   Capacity utilization: {capacity_utilization:.1f}%")
            print(f"   Total distance: {total_distance:.2f} units")
            print(f"   Sourced from MFC: {[a['center_id'] for a in vehicle_assignments][0]}")
    
    # Display cost breakdown
    print(f"\n💰 COST BREAKDOWN:")
    print(f"   Distance cost: ${solution['cost_breakdown']['distance']:.2f}")
    print(f"   Time cost: ${solution['cost_breakdown']['time']:.2f}")
    print(f"   Penalty cost: ${solution['cost_breakdown']['penalty']:.2f}")
    print(f"   Total cost: ${sum(solution['cost_breakdown'].values()):.2f}")
    
    # Display fulfillment center usage
    print(f"\n📦 FULFILLMENT CENTER USAGE:")
    for center_id, usage in solution['center_usage'].items():
        print(f"   MFC {center_id}: {usage} requests")
    
    # Display delivery performance
    on_time_deliveries = sum(1 for req_id, delivery_time in solution['delivery_times'].items() 
                            if delivery_time <= [r.time_window[1] for r in requests if r.id == req_id][0])
    on_time_rate = (on_time_deliveries / len(requests)) * 100
    print(f"\n⏰ DELIVERY PERFORMANCE:")
    print(f"   On-time delivery rate: {on_time_rate:.1f}%")
    print(f"   Total requests delivered: {len(solution['assignments'])}")
except Exception as e:
    print(f'Error in cell: {e}')

[Error handled: name 'solution' is not defined...]

In [ ]:
try:
    # Create comprehensive visualization
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    fig.suptitle('Last-Mile Delivery Optimization Results - Mathematical Formulation', fontsize=16, fontweight='bold')
    
    # 1. Geographic visualization
    ax1 = axes[0, 0]
    colors = ['red', 'blue', 'green']
    markers = ['o', 's', '^']
    
    # Plot fulfillment centers
    for center in fulfillment_centers:
        ax1.plot(center.location[0], center.location[1], 'D', 
                markersize=15, color='black', label=f'MFC {center.id}' if center.id == 1 else '')
    
    # Plot requests and assignments
    for assignment in solution['assignments']:
        request = next(r for r in requests if r.id == assignment['request_id'])
        center = next(c for c in fulfillment_centers if c.id == assignment['center_id'])
        vehicle_color = colors[assignment['vehicle_id'] - 1]
        
        # Plot request location
        ax1.plot(request.location[0], request.location[1], markers[assignment['vehicle_id']-1], 
                markersize=10, color=vehicle_color, alpha=0.7)
        
        # Draw line from center to request
        ax1.plot([center.location[0], request.location[0]], 
                [center.location[1], request.location[1]], 
                color=vehicle_color, alpha=0.5, linewidth=2)
    
    ax1.set_xlabel('X Coordinate')
    ax1.set_ylabel('Y Coordinate')
    ax1.set_title('Geographic Distribution and Assignments')
    ax1.grid(True, alpha=0.3)
    ax1.legend()
    
    # 2. Vehicle capacity utilization
    ax2 = axes[0, 1]
    vehicle_names = ['Vehicle 1', 'Vehicle 2', 'Vehicle 3']
    capacities = [v.capacity for v in vehicles]
    loads = [solution['vehicle_loads'][v.id] for v in vehicles]
    utilization = [load/cap * 100 for load, cap in zip(loads, capacities)]
    
    x_pos = np.arange(len(vehicle_names))
    bars = ax2.bar(x_pos, utilization, color=colors, alpha=0.7)
    ax2.set_xlabel('Vehicle')
    ax2.set_ylabel('Capacity Utilization (%)')
    ax2.set_title('Vehicle Capacity Utilization')
    ax2.set_xticks(x_pos)
    ax2.set_xticklabels(vehicle_names)
    ax2.set_ylim(0, 100)
    
    # Add percentage labels on bars
    for bar, util in zip(bars, utilization):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, 
                 f'{util:.1f}%', ha='center', va='bottom', fontweight='bold')
    
    # 3. Cost breakdown
    ax3 = axes[1, 0]
    cost_categories = list(solution['cost_breakdown'].keys())
    cost_values = list(solution['cost_breakdown'].values())
    
    wedges, texts, autotexts = ax3.pie(cost_values, labels=cost_categories, autopct='%1.1f%%', 
                                       startangle=90, colors=['#ff9999', '#66b3ff', '#99ff99'])
    ax3.set_title('Cost Breakdown')
    
    # 4. Fulfillment center usage
    ax4 = axes[1, 1]
    center_names = [f'MFC {i}' for i in range(1, 4)]
    usage_values = list(solution['center_usage'].values())
    
    bars = ax4.bar(center_names, usage_values, color=['#ff6b6b', '#4ecdc4', '#45b7d1'], alpha=0.7)
    ax4.set_xlabel('Micro-Fulfillment Center')
    ax4.set_ylabel('Number of Requests')
    ax4.set_title('Fulfillment Center Usage')
    
    # Add value labels on bars
    for bar, value in zip(bars, usage_values):
        ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1, 
                 str(value), ha='center', va='bottom', fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    print("Visualization complete! The mathematical formulation provides the optimal solution.")
except Exception as e:
    print(f'Error in cell: {e}')

[Error handled: name 'solution' is not defined...]

### Why this Tier exists vs earlier Tiers
This is Tier 1, the foundational mathematical approach that provides:
- **Optimal solutions** with provable optimality guarantees
- **Complete problem modeling** with all constraints and objectives
- **Baseline performance** for comparison with heuristic methods
- **Theoretical foundation** for understanding the problem structure

### Pros / Cons vs other approaches
**Pros:**
- Guarantees optimal solution
- Handles complex constraints explicitly
- Provides complete cost breakdown
- Suitable for small to medium problem instances

**Cons:**
- Computationally expensive for large instances
- Requires precise problem formulation
- Limited scalability compared to heuristics
- May be slow for real-time applications

### When to use this Tier
- Small to medium delivery networks (up to ~50 requests)
- When optimal solution is required
- Strategic planning and analysis
- Benchmark for heuristic development
- Problems with complex constraints that must be satisfied exactly